# 00 · Setup — Connections and Schema Creation

This notebook is the starting point for the ELT pipeline of the WWI Data Warehouse.

**What this notebook does:**
1. Defines the connection parameters for the two databases
2. Tests the connections (operational source and DW) — split per VPN state
3. Creates the `bronze`, `silver`, and `gold` schemas in the DW database
4. Creates the `bronze._load_control` table for ELT metadata
5. Creates the tables for the `gold` layer (dimensions and fact tables)
6. Validates the created structure

---
**VPN constraint** — the operational source (`postgres2.ipca.pt`) requires VPN ON, while Supabase is reachable only with VPN OFF. Cells that touch each are kept separate so you can toggle VPN between them.


## 1. Installation of dependencies

In [65]:
%pip install -r requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Connection parameters

Credentials are loaded from the `.env` file in the same folder.

In [66]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

# -- Operational Source -------------------------------------------------------
SRC_HOST     = os.getenv("SRC_HOST")
SRC_PORT     = int(os.getenv("SRC_PORT", 5432))
SRC_DB       = os.getenv("SRC_DB")
SRC_USER     = os.getenv("SRC_USER")
SRC_PASSWORD = os.getenv("SRC_PASSWORD")

# -- Data Warehouse (Supabase) ------------------------------------------------
SUPABASE_URL  = os.getenv("SUPABASE_URL")
SUPABASE_PORT = int(os.getenv("SUPABASE_PORT", 5432))
SUPABASE_DB   = os.getenv("SUPABASE_DB")
SUPABASE_USER = os.getenv("SUPABASE_USER")
SUPABASE_PASSWORD = os.getenv("SUPABASE_PASSWORD")

SCHEMAS = ["bronze", "silver", "gold"]

print(f"SRC: {SRC_USER}@{SRC_HOST}:{SRC_PORT}/{SRC_DB}")
print(f"Supabase: {SUPABASE_USER}@{SUPABASE_URL}:{SUPABASE_PORT}/{SUPABASE_DB}")


SRC: dss@postgres2.ipca.pt:5432/wwi
Supabase: postgres@db.tqhbvjlslhrjhtpjrygx.supabase.co:5432/postgres


## 3. Engine factory + DWH engine (run with VPN OFF)

Creates the SQLAlchemy engine pointing to Supabase. Run this cell with VPN OFF.

In [67]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

def make_engine(host, port, db, user, password, sslmode=None):
    return create_engine(URL.create("postgresql+psycopg2", username=user, password=password, host=host, port=port, database=db, query={"sslmode": sslmode} if sslmode else None))

if not all([SUPABASE_URL, SUPABASE_DB, SUPABASE_USER, SUPABASE_PASSWORD]):
    raise ValueError("SUPABASE_URL, SUPABASE_DB, SUPABASE_USER and SUPABASE_PASSWORD must be set in .env")
engine_dwh = make_engine(SUPABASE_URL, SUPABASE_PORT, SUPABASE_DB, SUPABASE_USER, SUPABASE_PASSWORD, sslmode="require")
print("✓ engine_dwh created.")


✓ engine_dwh created.


## 4. Source engine (run with VPN ON, optional)

Only needed if you want to test the source connection or run the validation query in section 8 (cell `## 8`). Skip this cell if you don't have VPN.

In [68]:
engine_src = make_engine(SRC_HOST, SRC_PORT, SRC_DB, SRC_USER, SRC_PASSWORD)
print("✓ engine_src created (requires VPN ON to actually connect).")


✓ engine_src created (requires VPN ON to actually connect).


## 5a. Test DWH connection (VPN OFF)

In [69]:
def test_connection(engine, label):
    try:
        with engine.connect() as conn:
            result = conn.execute(text("SELECT current_database(), version()"))
            db_name, version = result.fetchone()
            print(f"✓ {label}")
            print(f"  Database : {db_name}")
            print(f"  Version  : {version.split(',')[0]}")
    except Exception as e:
        print(f"✗ {label} — ERROR: {e}")

test_connection(engine_dwh, "Data Warehouse (Supabase)")


✗ Data Warehouse (Supabase) — ERROR: (psycopg2.OperationalError) could not translate host name "db.tqhbvjlslhrjhtpjrygx.supabase.co" to address: Name or service not known

(Background on this error at: https://sqlalche.me/e/20/e3q8)


## 5b. Test source connection (VPN ON)

Run this cell only when connected to the VPN.

In [70]:
test_connection(engine_src, "Operational Source (wwi)")


✓ Operational Source (wwi)
  Database : wwi
  Version  : PostgreSQL 17.9 on x86_64-pc-linux-gnu


## 6. Medallion architecture schemas (VPN OFF)

Creates the three schemas in the DW. Idempotent.

In [71]:
with engine_dwh.begin() as conn:
    for schema in SCHEMAS:
        conn.execute(text(f"CREATE SCHEMA IF NOT EXISTS {schema}"))
        print(f"✓ Schema '{schema}' created (or already exists)")

print("\nMedallion architecture schemas ready.")


OperationalError: (psycopg2.OperationalError) could not translate host name "db.tqhbvjlslhrjhtpjrygx.supabase.co" to address: Name or service not known

(Background on this error at: https://sqlalche.me/e/20/e3q8)

## 7. ELT metadata table `bronze._load_control` (VPN OFF)

Tracks every load attempt against bronze tables. Used by `01_bronze.ipynb` and `99_verification.py`. Created here so it exists even before the bronze pipeline runs the first time.

In [ ]:
DDL_CONTROL = """
CREATE TABLE IF NOT EXISTS bronze._load_control (
    table_name VARCHAR(100) NOT NULL,
    strategy VARCHAR(20) NOT NULL,
    snapshot_id INT,
    watermark_date DATE,
    loaded_at TIMESTAMP NOT NULL,
    rows_total INT,
    rows_inserted INT,
    rows_updated INT,
    rows_deleted INT,
    status VARCHAR(20) NOT NULL,
    PRIMARY KEY (table_name, loaded_at)
);
"""
with engine_dwh.begin() as conn:
    conn.execute(text(DDL_CONTROL))
print("✓ bronze._load_control ready.")


✓ bronze._load_control ready.


## 8. Creation of Gold layer tables (VPN OFF)

**Dimensions:** DimEmployee, DimCustomer, DimLocation, DimDate, DimProduct, DimDeliveryMethod, DimPaymentMethod.

**Fact tables:** FactSales (line level), FactInvoices (header level).

All SCD Type 2 dimensions carry `version`, `date_from`, `date_to`. The current row has `date_to IS NULL`.

In [ ]:
DDL_GOLD = """
-- DimEmployee
CREATE TABLE IF NOT EXISTS gold.DimEmployee (
    EmployeeKey SERIAL PRIMARY KEY,
    version INT NOT NULL,
    date_from TIMESTAMP NOT NULL,
    date_to TIMESTAMP,
    PersonID INT NOT NULL,
    FullName VARCHAR(255),
    IsSalesperson INT
);

-- DimCustomer
CREATE TABLE IF NOT EXISTS gold.DimCustomer (
    CustomerKey SERIAL PRIMARY KEY,
    version INT NOT NULL,
    date_from TIMESTAMP NOT NULL,
    date_to TIMESTAMP,
    CustomerID INT NOT NULL,
    CustomerName VARCHAR(255),
    Category VARCHAR(255)
);

-- DimLocation
CREATE TABLE IF NOT EXISTS gold.DimLocation (
    LocationKey SERIAL PRIMARY KEY,
    version INT NOT NULL,
    date_from TIMESTAMP NOT NULL,
    date_to TIMESTAMP,
    LocationID INT NOT NULL,
    City VARCHAR(255),
    State VARCHAR(255),
    Country VARCHAR(255),
    SalesTerritory VARCHAR(50)
);

-- DimDate
CREATE TABLE IF NOT EXISTS gold.DimDate (
    DateKey INT PRIMARY KEY,
    Date DATE,
    Year INT,
    Month INT,
    Day INT
);

-- DimProduct
CREATE TABLE IF NOT EXISTS gold.DimProduct (
    ProductKey SERIAL PRIMARY KEY,
    version INT NOT NULL,
    date_from TIMESTAMP NOT NULL,
    date_to TIMESTAMP,
    StockItemID INT NOT NULL,
    StockItemName VARCHAR(255),
    Brand VARCHAR(255)
);

-- DimDeliveryMethod
CREATE TABLE IF NOT EXISTS gold.DimDeliveryMethod (
    DeliveryMethodKey SERIAL PRIMARY KEY,
    version INT NOT NULL,
    date_from TIMESTAMP NOT NULL,
    date_to TIMESTAMP,
    DeliveryMethodID INT NOT NULL,
    DeliveryMethodName VARCHAR(255)
);

-- DimPaymentMethod
CREATE TABLE IF NOT EXISTS gold.DimPaymentMethod (
    PaymentMethodKey SERIAL PRIMARY KEY,
    version INT NOT NULL,
    date_from TIMESTAMP NOT NULL,
    date_to TIMESTAMP,
    PaymentMethodID INT NOT NULL,
    PaymentMethodName VARCHAR(255)
);

-- FactSales (Line Level)
CREATE TABLE IF NOT EXISTS gold.FactSales (
    SalesKey SERIAL PRIMARY KEY,
    InvoiceID INT,
    DateKey INT,
    EmployeeKey INT,
    CustomerKey INT,
    ProductKey INT,
    LocationKey INT,
    Quantity INT,
    UnitPrice NUMERIC(10,2),
    TaxAmount NUMERIC(10,2),
    ExtendedPrice NUMERIC(10,2),
    LineProfit NUMERIC(10,2),
    FOREIGN KEY (DateKey) REFERENCES gold.DimDate(DateKey),
    FOREIGN KEY (EmployeeKey) REFERENCES gold.DimEmployee(EmployeeKey),
    FOREIGN KEY (CustomerKey) REFERENCES gold.DimCustomer(CustomerKey),
    FOREIGN KEY (ProductKey) REFERENCES gold.DimProduct(ProductKey),
    FOREIGN KEY (LocationKey) REFERENCES gold.DimLocation(LocationKey)
);

-- FactInvoices (Header Level)
CREATE TABLE IF NOT EXISTS gold.FactInvoices (
    InvoiceFactKey SERIAL PRIMARY KEY,
    InvoiceID INT,
    DateKey INT,
    EmployeeKey INT,
    CustomerKey INT,
    LocationKey INT,
    AccountsEmployeeKey INT,
    DeliveryMethodKey INT,
    InvoiceAmount NUMERIC(10,2),
    PaymentDelay_Days INT,
    OutstandingBalance NUMERIC(10,2),
    FOREIGN KEY (DateKey) REFERENCES gold.DimDate(DateKey),
    FOREIGN KEY (EmployeeKey) REFERENCES gold.DimEmployee(EmployeeKey),
    FOREIGN KEY (CustomerKey) REFERENCES gold.DimCustomer(CustomerKey),
    FOREIGN KEY (LocationKey) REFERENCES gold.DimLocation(LocationKey),
    FOREIGN KEY (AccountsEmployeeKey) REFERENCES gold.DimEmployee(EmployeeKey),
    FOREIGN KEY (DeliveryMethodKey) REFERENCES gold.DimDeliveryMethod(DeliveryMethodKey)
);
"""

with engine_dwh.begin() as conn:
    conn.execute(text(DDL_GOLD))

print("✓ Gold layer tables successfully created.")


✓ Gold layer tables successfully created.


## 9. Validation — Inventory of created schemas and tables (VPN OFF)

In [ ]:
import pandas as pd

query = """
    SELECT table_schema AS schema, table_name, table_type AS type
    FROM information_schema.tables
    WHERE table_schema IN ('bronze', 'silver', 'gold')
    ORDER BY table_schema, table_name
"""
with engine_dwh.connect() as conn:
    df = pd.read_sql(text(query), conn)
print("Objects created in the DW:")
print(df.to_string(index=False))


Objects created in the DW:
schema        table_name       type
bronze     _load_control BASE TABLE
  gold       dimcustomer BASE TABLE
  gold           dimdate BASE TABLE
  gold dimdeliverymethod BASE TABLE
  gold       dimemployee BASE TABLE
  gold       dimlocation BASE TABLE
  gold  dimpaymentmethod BASE TABLE
  gold        dimproduct BASE TABLE
  gold      factinvoices BASE TABLE
  gold         factsales BASE TABLE


## 10. Validation — Operational source tables (VPN ON)

Run this cell only when connected to the VPN.

In [ ]:
import pandas as pd

query_src = """
    SELECT t.table_name,
           (SELECT COUNT(*) FROM information_schema.columns c
             WHERE c.table_name = t.table_name AND c.table_schema = 'public') AS columns
    FROM information_schema.tables t
    WHERE t.table_schema = 'public' AND t.table_type = 'BASE TABLE'
    ORDER BY t.table_name
"""
with engine_src.connect() as conn:
    df_src = pd.read_sql(text(query_src), conn)
print(f"Tables in operational source ({SRC_DB}):")
print(df_src.to_string(index=False))


Tables in operational source (wwi):
          table_name  columns
        buyinggroups        2
              cities        5
              colors        2
           countries        8
  customercategories        2
           customers       28
customertransactions       12
     deliverymethods        2
        invoicelines       11
            invoices       23
          orderlines        9
              orders       14
        packagetypes        2
      paymentmethods        2
              people       18
        specialdeals       12
      stateprovinces        6
         stockgroups        2
          stockitems       22
    transactiontypes        2


## 11. Summary

| Step | VPN | Status |
|---|---|---|
| Connection to DW | OFF | ✓ |
| Connection to source | ON  | ✓ |
| `bronze` / `silver` / `gold` schemas | OFF | ✓ |
| `bronze._load_control` | OFF | ✓ |
| Gold tables | OFF | ✓ |

**Next step:** execute `01_bronze.ipynb`.